# 03 - 解析 SEC 财报 HTML

上一节已经把 SEC 财报下载到了本地。这一节将原始 HTML 转换为适合后续检索和大模型分析的纯文本。

本节的数据流：

```text
本地 HTML -> BeautifulSoup 解析 -> 删除网页噪声 -> 提取纯文本 -> 保存 TXT
```

## 1. 导入工具

`Path` 用来处理文件路径，`BeautifulSoup` 用来理解 HTML 的标签结构。

In [ ]:
import re
from pathlib import Path

from bs4 import BeautifulSoup

DATA_DIR = Path("data/sec")

## 2. 找到下载的财报

这里默认读取 `AAPL` 目录中最新的 `10-K` HTML。修改 `TICKER` 和 `FORM_TYPE` 就可以处理其他公司或季度报告。

In [ ]:
TICKER = "AAPL"
FORM_TYPE = "10-K"

filing_files = sorted(
    (DATA_DIR / TICKER).glob(f"*_{FORM_TYPE}_*.htm*"),
    reverse=True,
)

if not filing_files:
    raise FileNotFoundError(
        f"没有找到 {TICKER} 的 {FORM_TYPE}，请先运行 02_sec_filing.ipynb"
    )

filing_path = filing_files[0]
print(f"读取文件：{filing_path}")
print(f"文件大小：{filing_path.stat().st_size / 1024 / 1024:.2f} MB")

## 3. 读取和解析 HTML

HTML 不只是正文，还包含标签、脚本、样式和 XBRL 数据。`BeautifulSoup` 会把这些内容转换成一棵可以查询和修改的结构树。

In [ ]:
html = filing_path.read_bytes()
soup = BeautifulSoup(html, "html.parser")

print(f"原始 HTML 字节数：{len(html):,}")
print(f"网页标题：{soup.title.get_text(strip=True) if soup.title else '未找到'}")

## 4. 清理网页噪声并提取文本

我们删除脚本、样式、导航等不属于财报正文的内容，但暂时保留表格，因为财务数据通常位于表格中。

In [ ]:
def html_to_clean_text(html: bytes) -> str:
    soup = BeautifulSoup(html, "html.parser")

    # 删除不会出现在可读财报正文中的网页元素。
    for tag in soup.find_all(
        ["script", "style", "noscript", "nav", "ix:header"]
    ):
        tag.decompose()

    # SEC Inline XBRL 可能包含隐藏的重复数据。
    for tag in soup.find_all(
        lambda element: element.name
        and element.name.lower().endswith("hidden")
    ):
        tag.decompose()

    raw_text = soup.get_text(separator="\n")
    clean_lines = []

    for line in raw_text.splitlines():
        line = re.sub(r"\s+", " ", line).strip()
        if line and (not clean_lines or line != clean_lines[-1]):
            clean_lines.append(line)

    return "\n".join(clean_lines)


clean_text = html_to_clean_text(html)
print(f"清理后的字符数：{len(clean_text):,}")
print(f"清理后的行数：{len(clean_text.splitlines()):,}")

## 5. 预览清理结果

先看前 3,000 个字符，确认内容已经从 HTML 标签变成可读文本。

In [ ]:
print(clean_text[:3000])

## 6. 尝试定位财报章节

10-K 常见章节包括 `Item 1. Business`、`Item 1A. Risk Factors` 和 `Item 7. Management's Discussion and Analysis`。这里先搜索关键词，观察它们在文本中的位置。

In [ ]:
def show_keyword_context(text: str, keyword: str, context: int = 300) -> None:
    position = text.lower().find(keyword.lower())

    if position == -1:
        print(f"没有找到关键词：{keyword}")
        return

    start = max(0, position - context)
    end = min(len(text), position + len(keyword) + context)
    print(f"关键词位置：{position:,}\n")
    print(text[start:end])


show_keyword_context(clean_text, "Item 1.")

## 7. 保存纯文本

把清理结果保存为与原财报同名的 `.txt` 文件。下一节可以直接读取它，不必每次重新解析 HTML。

In [ ]:
text_path = filing_path.with_suffix(".txt")
text_path.write_text(clean_text, encoding="utf-8")

print(f"纯文本已保存：{text_path.resolve()}")
print(f"文本文件大小：{text_path.stat().st_size / 1024:.2f} KB")

## 这一节完成了什么

现在我们已经把 SEC 原始 HTML 转换成了可以被程序搜索、切块和交给 DeepSeek 阅读的纯文本。

下一步将解决一个重要问题：一份 10-K 太长，不能直接全部发送给模型。因此需要先提取章节，再把章节切成大小合适的文本块。